# LAnoBERT Kaggle - GitHub từng bước

Notebook clone trực tiếp `https://github.com/rubyhcm/lv-new.git`, tìm thư mục `LAnoBERT`, rồi chạy từng bước trên Kaggle GPU.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys, os

GITHUB_URL = 'https://github.com/rubyhcm/lv-new.git'
GITHUB_BRANCH = 'main'
CLONE_DIR = Path('/kaggle/working/lv-new')
PROJECT = Path('/kaggle/working/LAnoBERT')
CONFIG = 'configs/bgl.yaml'
DOWNLOAD_BGL = False
EPOCHS = 1
MAX_EVAL_SAMPLES = 10000
print('GitHub:', GITHUB_URL)
print('Working clone:', CLONE_DIR)


In [ ]:
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Hãy bật Accelerator = GPU trong Kaggle.')


In [ ]:
os.chdir('/kaggle/working')
if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_URL, str(CLONE_DIR)], check=True)
candidates = [p for p in CLONE_DIR.rglob('lanobert') if p.is_dir() and (p.parent / 'configs').is_dir()]
assert len(candidates) == 1, f'Không tìm thấy duy nhất LAnoBERT: {candidates}'
PROJECT = candidates[0].parent
print('Project:', PROJECT)
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip())


In [ ]:
%pip install -q -r {PROJECT / 'requirements.txt'} pytest

In [ ]:
os.chdir(PROJECT)
print('Current directory:', Path.cwd())
print('Config:', Path(CONFIG).exists())
print('BGL directory:', Path('data/BGL').exists())
if Path('data/BGL').exists():
    print(*sorted(str(p) for p in Path('data/BGL').iterdir()), sep='\n')


In [ ]:
PROJECT = Path(PROJECT).resolve()
os.chdir(PROJECT)
raw_log = PROJECT / 'data/BGL/BGL.log'
if not raw_log.exists():
    candidates = [p for p in Path('/kaggle/input').rglob('BGL.log') if p.is_file()]
    if candidates:
        source_bgl = candidates[0].parent
        target_bgl = PROJECT / 'data/BGL'
        target_bgl.mkdir(parents=True, exist_ok=True)
        for source in source_bgl.iterdir():
            target = target_bgl / source.name
            if source.is_file() and not target.exists():
                shutil.copy2(source, target)
        print('Copied BGL from:', source_bgl)
    elif DOWNLOAD_BGL:
        subprocess.run(['bash', 'scripts/download_data.sh', 'bgl'], cwd=PROJECT, check=True)
bgl_candidates = list(Path('/kaggle/input').rglob('BGL.log'))
assert raw_log.exists(), f"Thiếu {raw_log}; các file BGL tìm thấy: {bgl_candidates}"
print(raw_log, f'{raw_log.stat().st_size / (1024**3):.2f} GB')


## Kiểm tra mã nguồn

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-v'], check=True)
subprocess.run(['bash', '-n', 'scripts/ensure_data.sh', 'scripts/run_pipeline.sh'], check=True)
print('Tests and shell checks passed')


## Split và preprocess BGL

In [ ]:
subprocess.run(['bash', 'scripts/ensure_data.sh', CONFIG], check=True)

In [ ]:
for name in ['BGL_train_normal.raw', 'BGL_test.raw', 'BGL_test_label.log', 'BGL_train_normal_parsed.log', 'BGL_test_parsed.log']:
    path = Path('data/BGL') / name
    print(f'{path}: {path.stat().st_size / (1024**2):.1f} MB' if path.exists() else f'MISSING: {path}')


## Train tokenizer

In [ ]:
subprocess.run([sys.executable, '-m', 'lanobert.tokenizer', '--config', CONFIG], check=True)
print('Tokenizer ready:', Path('outputs/BGL/tokenizer').exists())


## Tạo cấu hình Kaggle

In [ ]:
import yaml
with open(CONFIG, encoding='utf-8') as handle:
    runtime = yaml.safe_load(handle)
runtime.setdefault('train', {})['num_train_epochs'] = EPOCHS
runtime.setdefault('inference', {})['max_eval_samples'] = MAX_EVAL_SAMPLES
with open('configs/kaggle_runtime.yaml', 'w', encoding='utf-8') as handle:
    yaml.safe_dump(runtime, handle, sort_keys=False)
print(Path('configs/kaggle_runtime.yaml').read_text())


## Train LAnoBERT baseline

In [ ]:
subprocess.run([sys.executable, '-m', 'lanobert.train', '--config', 'configs/kaggle_runtime.yaml'], check=True)

## Inference và xem kết quả

In [ ]:
subprocess.run([sys.executable, '-m', 'lanobert.inference', '--config', 'configs/kaggle_runtime.yaml'], check=True)
result_dir = Path('outputs/BGL/results')
print(*sorted(str(p) for p in result_dir.iterdir()), sep='\n')


In [ ]:
for report in sorted(Path('outputs/BGL/results').glob('*report.txt')):
    print(f'\n===== {report.name} =====\n{report.read_text()[:3000]}')
archive = shutil.make_archive('/kaggle/working/lanobert-results', 'zip', 'outputs')
print('Artifact:', archive)
